# Export Validated Report

Generate a sanitized report from experiment runs and download it.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = (
    "google.colab" in sys.modules
    or bool(os.environ.get("COLAB_RELEASE_TAG"))
    or bool(os.environ.get("COLAB_BACKEND_VERSION"))
)
REPO_BRANCH = "test/colab-runtime-validation"
REPO_URL = "https://github.com/error-wtf/counterexample-commons.git"
COLAB_REPO_DIR = Path("/content/counterexample-commons")

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
             REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "checkout", "-B",
             REPO_BRANCH, f"origin/{REPO_BRANCH}"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(COLAB_REPO_DIR)],
        check=True,
    )
    repo_root = COLAB_REPO_DIR
else:
    start = Path.cwd().resolve()
    repo_root = next(
        (p for p in [start, *start.parents]
         if (p / "counterexample_commons").is_dir()
         and (p / "pyproject.toml").is_file()),
        None,
    )
    if repo_root is None:
        raise RuntimeError(
            "Counterexample Commons repository root not found. "
            "Run this notebook from inside a cloned repository checkout."
        )

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import counterexample_commons
print(f"Repository root: {repo_root}")
print(f"Counterexample Commons version: {counterexample_commons.__version__}")
print(
    "Validation status: local execution or test-branch bootstrap only; "
    "fresh Colab runtime PASS requires manual confirmation."
)


In [ ]:
import json
from counterexample_commons.validators import validate_grid_configuration
from counterexample_commons.experiments.sanitization import (
    sanitize_for_export, is_safe_for_export,
)

print("Finite validation export only; not an asymptotic theorem or Sawin reproduction.")
result = validate_grid_configuration(5)
export_data = {
    "configuration": result["configuration"],
    "n": result["n"],
    "actual_edges": result["actual_edges"],
    "status": result["status"],
    "pass": result["pass"],
    "scope": "LOCALLY_REPRODUCED_EXACT_FINITE_RESULT",
    "sawin_status": "SOURCE_DOCUMENTED — not locally reproduced",
}
export_text = json.dumps(export_data, indent=2)
safe, reason = is_safe_for_export(export_text)
print(f"Safe for export: {safe} ({reason})")
if safe:
    sanitized = sanitize_for_export(export_text)
    print("Sanitized export:")
    print(sanitized)
